# 12 — BRD Gap Analysis (4 hạng mục ưu tiên)

Bổ sung các phân tích còn thiếu trong BRD v1.1 (mục 3.4) trên `hotel_bookings_v5.csv`:

1. **Interaction 3 chiều** — Lead time × Segment × Mùa vụ
2. **Revenue Loss** — Doanh thu tiềm năng mất do hủy phòng
3. **Room Mis-match** — Upgrade vs Downgrade (median ADR)
4. **Deposit Simulation** — Mô phỏng chính sách cọc Online TA

> Báo cáo chi tiết: `reports/12_brd_gap_analysis.md` · BRD v1.2: `reports/12_brd_v1_2.md`


In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (12, 5)

NOTEBOOK_DIR = Path(os.environ.get("VSCODE_NOTEBOOK_DIR", Path.cwd()))
ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / "data").is_dir() else NOTEBOOK_DIR

DATA_PATH = ROOT / "data" / "hotel_bookings_v5.csv"

PEAK_MONTHS = {"July", "August"}
DEPOSIT_CANCEL_REDUCTION = 0.30
DEPOSIT_VOLUME_REDUCTION = 0.10

In [ ]:
def fmt_int(n: float) -> str:
    return f"{int(round(n)):,}".replace(",", ".")


def fmt_pct(x: float, d: int = 2) -> str:
    return f"{x * 100:.{d}f}".replace(".", ",")


def fmt_eur(x: float, d: int = 2) -> str:
    return f"{x:,.{d}f} €".replace(",", "X").replace(".", ",").replace("X", ".")


def load_data() -> pd.DataFrame:
    df = pd.read_csv(DATA_PATH)
    df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]
    df["booking_revenue"] = df["adr"].clip(lower=0) * df["total_nights"].clip(lower=0)
    df["is_peak"] = df["arrival_date_month"].isin(PEAK_MONTHS)
    return df


def build_room_adr_rank(stay: pd.DataFrame):
    reserved_med = stay.groupby("reserved_room_type", observed=True)["adr"].median()
    assigned_med = stay.groupby("assigned_room_type", observed=True)["adr"].median()
    counts = stay.groupby("reserved_room_type", observed=True)["adr"].count()
    all_types = sorted(set(reserved_med.index) | set(assigned_med.index))
    medians = {}
    for rt in all_types:
        medians[rt] = float(reserved_med[rt]) if rt in reserved_med.index else float(assigned_med[rt])
    ordered = pd.Series(medians).sort_values()
    rank_map = {rt: i + 1 for i, rt in enumerate(ordered.index)}
    table = pd.DataFrame({
        "median_adr": ordered.values,
        "rank": [rank_map[rt] for rt in ordered.index],
        "bookings_reserved": [int(counts.get(rt, 0)) for rt in ordered.index],
    }, index=ordered.index)
    return table, rank_map, medians


def get_cheap_reserved_types(stay: pd.DataFrame, n: int = 2) -> list[str]:
    return (
        stay.groupby("reserved_room_type", observed=True)["adr"]
        .median().sort_values().head(n).index.tolist()
    )


def map_room_rank(series: pd.Series, rank_map: dict[str, int]) -> pd.Series:
    return series.map(rank_map).fillna(0).astype(int)


def inspect_room_rank(stay: pd.DataFrame) -> dict:
    table, rank_map, medians = build_room_adr_rank(stay)
    stay_mm = stay[stay["reserved_room_type"] != stay["assigned_room_type"]].copy()
    stay_mm["r_med"] = stay_mm["reserved_room_type"].map(medians)
    stay_mm["a_med"] = stay_mm["assigned_room_type"].map(medians)
    adr_dir = stay_mm.apply(
        lambda r: "upgrade" if r["a_med"] > r["r_med"]
        else ("downgrade" if r["a_med"] < r["r_med"] else "lateral"),
        axis=1,
    )
    return {
        "table": table,
        "rank_map": rank_map,
        "cheap_room_types": get_cheap_reserved_types(stay),
        "alpha_order": "".join(sorted(rank_map.keys())),
        "adr_order": "".join(table.index),
        "alphabet_matches_adr": "".join(sorted(rank_map.keys())) == "".join(table.index),
        "adr_direction_counts": adr_dir.value_counts(),
    }

In [ ]:
def analyze_interaction_3d(df: pd.DataFrame) -> dict:
    canceled = df["is_canceled"] == 1
    total_loss = df.loc[canceled, "booking_revenue"].sum()
    mask = (df["market_segment"] == "Online TA") & (df["lead_time"] > 90) & df["is_peak"]
    seg = df.loc[mask]
    seg_loss = seg.loc[seg["is_canceled"] == 1, "booking_revenue"].sum()
    mask_br = (
        (df["market_segment"] == "Online TA")
        & (df["distribution_channel"] == "TA/TO")
        & (df["lead_time"] > 60) & df["is_peak"]
    )
    br = df.loc[mask_br]
    br_loss = br.loc[br["is_canceled"] == 1, "booking_revenue"].sum()
    return {
        "bookings": len(seg), "canceled": int(seg["is_canceled"].sum()),
        "cancel_rate": seg["is_canceled"].mean() if len(seg) else 0.0,
        "revenue_loss": seg_loss,
        "pct_total_loss": seg_loss / total_loss if total_loss else 0.0,
        "mean_adr_canceled": seg.loc[seg["is_canceled"] == 1, "adr"].mean(),
        "br_bookings": len(br),
        "br_cancel_rate": br["is_canceled"].mean() if len(br) else 0.0,
        "br_revenue_loss": br_loss,
        "br_pct_total_loss": br_loss / total_loss if total_loss else 0.0,
    }


def analyze_revenue_loss(df: pd.DataFrame) -> dict:
    canceled = df["is_canceled"] == 1
    loss_df = df.loc[canceled & (df["booking_revenue"] > 0)].copy()
    by_month = (
        loss_df.groupby("arrival_date_month", observed=True)["booking_revenue"]
        .agg(["sum", "count", "mean"])
        .reindex(["January", "February", "March", "April", "May", "June",
                  "July", "August", "September", "October", "November", "December"])
    )
    peak_loss = loss_df.loc[loss_df["is_peak"], "booking_revenue"]
    total_loss = loss_df["booking_revenue"].sum()
    peak_nights = loss_df.loc[loss_df["is_peak"], "total_nights"].sum()
    return {
        "total_loss": total_loss,
        "realized_revenue": df.loc[~canceled, "booking_revenue"].sum(),
        "potential_revenue": df["booking_revenue"].sum(),
        "loss_pct_of_potential": total_loss / df["booking_revenue"].sum(),
        "canceled_bookings": len(loss_df),
        "mean_loss_per_booking": loss_df["booking_revenue"].mean(),
        "mean_loss_per_night": (loss_df["adr"] * loss_df["total_nights"]).sum() / loss_df["total_nights"].sum(),
        "peak_mean_loss_per_night": peak_loss.sum() / peak_nights if peak_nights else 0,
        "by_month": by_month,
        "by_hotel": loss_df.groupby("hotel", observed=True)["booking_revenue"].agg(["sum", "count"]),
    }


def analyze_room_mismatch(df: pd.DataFrame) -> dict:
    stay = df[(df["is_canceled"] == 0) & (df["adr"] > 0)].copy()
    rank_table, rank_map, medians = build_room_adr_rank(stay)
    cheap_types = get_cheap_reserved_types(stay)
    stay["room_match"] = stay["reserved_room_type"] == stay["assigned_room_type"]
    stay["reserved_median_adr"] = stay["reserved_room_type"].map(medians)
    stay["assigned_median_adr"] = stay["assigned_room_type"].map(medians)
    stay["reserved_rank"] = map_room_rank(stay["reserved_room_type"], rank_map)
    stay["assigned_rank"] = map_room_rank(stay["assigned_room_type"], rank_map)
    stay["rank_delta"] = stay["assigned_rank"] - stay["reserved_rank"]
    mm = stay[~stay["room_match"]].copy()
    mm["shift_type"] = np.select(
        [mm["assigned_median_adr"] > mm["reserved_median_adr"],
         mm["assigned_median_adr"] < mm["reserved_median_adr"],
         mm["assigned_median_adr"] == mm["reserved_median_adr"]],
        ["upgrade", "downgrade", "lateral"], default="unknown",
    )
    free_upgrade = mm[mm["reserved_room_type"].isin(cheap_types) & (mm["shift_type"] == "upgrade")]
    matched = stay[stay["room_match"]]
    summary = mm.groupby("shift_type", observed=True).agg(
        bookings=("adr", "count"), mean_adr=("adr", "mean"), mean_rank_delta=("rank_delta", "mean"),
    )
    return {
        "mismatch_rate": 1 - stay["room_match"].mean(),
        "match_mean_adr": matched["adr"].mean(),
        "mismatch_mean_adr": mm["adr"].mean(),
        "adr_gap": matched["adr"].mean() - mm["adr"].mean(),
        "summary": summary, "room_rank_table": rank_table, "cheap_room_types": cheap_types,
        "free_upgrade_count": len(free_upgrade),
        "free_upgrade_pct_of_mismatch": len(free_upgrade) / len(mm) if len(mm) else 0,
        "upgrade_count": int((mm["shift_type"] == "upgrade").sum()),
        "downgrade_count": int((mm["shift_type"] == "downgrade").sum()),
        "lateral_count": int((mm["shift_type"] == "lateral").sum()),
        "mm": mm,
    }


def analyze_deposit_simulation(df: pd.DataFrame) -> dict:
    seg = df.loc[(df["market_segment"] == "Online TA") & (df["lead_time"] > 30)].copy()
    cancel_rate = seg["is_canceled"].mean()
    baseline_net = seg.loc[seg["is_canceled"] == 0, "booking_revenue"].sum()
    scenario_bookings = len(seg) * (1 - DEPOSIT_VOLUME_REDUCTION)
    scenario_cancel_rate = cancel_rate * (1 - DEPOSIT_CANCEL_REDUCTION)
    scenario_net = scenario_bookings * seg["booking_revenue"].mean() * (1 - scenario_cancel_rate)
    deposit_retained = scenario_bookings * (1 - scenario_cancel_rate) * seg["adr"].mean() * 0.8
    return {
        "bookings": len(seg), "cancel_rate": cancel_rate, "baseline_net": baseline_net,
        "scenario_bookings": scenario_bookings, "scenario_cancel_rate": scenario_cancel_rate,
        "scenario_net": scenario_net, "net_change": scenario_net - baseline_net,
        "net_change_pct": (scenario_net - baseline_net) / baseline_net if baseline_net else 0,
        "scenario_net_with_deposit": scenario_net + deposit_retained,
        "net_change_with_deposit": scenario_net + deposit_retained - baseline_net,
    }

In [ ]:
def plot_cancel_rate_hotspots(i3d: dict, df: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    labels = [
        "Online TA\nlead>90 Jul-Aug",
        "Online TA×TA/TO\nlead>60 Jul-Aug",
        "Toàn hệ thống",
        "Online TA\n(tổng)",
    ]
    rates = [
        i3d["cancel_rate"] * 100,
        i3d["br_cancel_rate"] * 100,
        df["is_canceled"].mean() * 100,
        df.loc[df["market_segment"] == "Online TA", "is_canceled"].mean() * 100,
    ]
    ax.bar(labels, rates, color=["#C0392B", "#E67E22", "#95A5A6", "#F39C12"])
    ax.set_ylabel("Tỷ lệ hủy (%)")
    ax.set_title("So sánh tỷ lệ hủy — tổ hợp rủi ro BRD")
    for i, v in enumerate(rates):
        ax.text(i, v + 0.8, f"{v:.1f}%", ha="center", fontsize=10)
    fig.tight_layout()
    plt.show()


def plot_revenue_loss_by_month(rev: dict) -> None:
    fig, ax = plt.subplots(figsize=(12, 5))
    m = rev["by_month"].dropna(subset=["sum"])
    ax.bar(m.index, m["sum"] / 1e6, color="#E74C3C", alpha=0.85)
    ax.set_title("Doanh thu tiềm năng mất do hủy phòng theo tháng đến")
    ax.set_ylabel("Triệu €")
    plt.xticks(rotation=45, ha="right")
    fig.tight_layout()
    plt.show()


def plot_room_mismatch(mm: dict) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    st = mm["summary"]
    bar_colors = {"upgrade": "#27AE60", "downgrade": "#E74C3C", "lateral": "#3498DB"}
    axes[0].bar(st.index, st["bookings"], color=[bar_colors.get(i, "#95A5A6") for i in st.index])
    axes[0].set_title("Số booking không khớp phòng theo loại chuyển")
    axes[0].set_ylabel("Booking")
    sns.boxplot(data=mm["mm"], x="shift_type", y="adr", ax=axes[1], order=["upgrade", "lateral", "downgrade"])
    axes[1].set_title("ADR theo loại chuyển phòng (mis-match)")
    axes[1].set_xlabel("")
    fig.tight_layout()
    plt.show()


def plot_deposit_simulation(dep: dict) -> None:
    fig, ax = plt.subplots(figsize=(8, 5))
    scenarios = ["Hiện tại", "Sau cọc\n(chỉ net stay)", "Sau cọc\n(+ cọc giữ lại)"]
    values = [dep["baseline_net"] / 1e6, dep["scenario_net"] / 1e6, dep["scenario_net_with_deposit"] / 1e6]
    bars = ax.bar(scenarios, values, color=["#95A5A6", "#3498DB", "#27AE60"])
    ax.set_ylabel("Net Revenue (triệu €)")
    ax.set_title("Mô phỏng Deposit Policy — Online TA, lead_time > 30 ngày")
    for b, v in zip(bars, values):
        ax.text(b.get_x() + b.get_width() / 2, v + 0.3, f"{v:.1f}M", ha="center")
    fig.tight_layout()
    plt.show()

In [ ]:
df = load_data()
print(f"Đọc: {DATA_PATH.name} — {len(df):,} booking")

## 1. Interaction 3 chiều (Lead time × Segment × Mùa vụ)


In [ ]:
i3d = analyze_interaction_3d(df)
display(pd.DataFrame([
    {"Tổ hợp": "Online TA + lead>90 + Jul-Aug", "Booking": i3d["bookings"],
     "Tỷ lệ hủy": f"{i3d['cancel_rate']*100:.2f}%",
     "Doanh thu mất": fmt_eur(i3d["revenue_loss"], 0),
     "% tổng mất": f"{i3d['pct_total_loss']*100:.2f}%"},
    {"Tổ hợp": "Online TA×TA/TO + lead>60 + Jul-Aug", "Booking": i3d["br_bookings"],
     "Tỷ lệ hủy": f"{i3d['br_cancel_rate']*100:.2f}%",
     "Doanh thu mất": fmt_eur(i3d["br_revenue_loss"], 0),
     "% tổng mất": f"{i3d['br_pct_total_loss']*100:.2f}%"},
]))
plot_cancel_rate_hotspots(i3d, df)

## 2. Revenue Loss


In [ ]:
rev = analyze_revenue_loss(df)
print(f"Tổng doanh thu mất: {fmt_eur(rev['total_loss'], 0)}")
print(f"Tỷ lệ mất / tiềm năng: {rev['loss_pct_of_potential']*100:.2f}%")
print(f"Mean mất/đêm Jul-Aug: {fmt_eur(rev['peak_mean_loss_per_night'])}")
display(rev["by_month"][["sum", "count"]].dropna().assign(sum_M=lambda x: (x["sum"] / 1e6).round(2)))
plot_revenue_loss_by_month(rev)

## Kiểm tra thứ hạng phòng (median ADR)

Thứ tự A→B→C… **không** khớp giá thực tế. Xếp hạng động từ median ADR trên `reserved_room_type`.


In [ ]:
stay_check = df[(df["is_canceled"] == 0) & (df["adr"] > 0)]
rank_info = inspect_room_rank(stay_check)
print(f"Alphabet: {rank_info['alpha_order']} | Median ADR: {rank_info['adr_order']}")
print(f"Khớp? {rank_info['alphabet_matches_adr']} | Hạng rẻ: {rank_info['cheap_room_types']}")
print(rank_info["adr_direction_counts"].to_string())
display(rank_info["table"].round(2))

## 3. Room Mis-match — Upgrade vs Downgrade


In [ ]:
mm = analyze_room_mismatch(df)
print(f"Mis-match: {mm['mismatch_rate']*100:.2f}% | ADR gap: {fmt_eur(mm['adr_gap'])}")
print(f"Hạng rẻ: {mm['cheap_room_types']} | Free upgrade: {fmt_int(mm['free_upgrade_count'])} ({mm['free_upgrade_pct_of_mismatch']*100:.1f}%)")
display(mm["summary"].round(2))
plot_room_mismatch(mm)

## 4. Deposit Simulation


In [ ]:
dep = analyze_deposit_simulation(df)
display(pd.DataFrame([
    {"Kịch bản": "Hiện tại", "Booking": int(dep["bookings"]),
     "Tỷ lệ hủy": f"{dep['cancel_rate']*100:.2f}%", "Net Revenue": fmt_eur(dep["baseline_net"], 0)},
    {"Kịch bản": "Sau cọc (lưu trú)", "Booking": int(dep["scenario_bookings"]),
     "Tỷ lệ hủy": f"{dep['scenario_cancel_rate']*100:.2f}%", "Net Revenue": fmt_eur(dep["scenario_net"], 0)},
    {"Kịch bản": "Sau cọc (+ cọc)", "Booking": int(dep["scenario_bookings"]),
     "Tỷ lệ hủy": f"{dep['scenario_cancel_rate']*100:.2f}%", "Net Revenue": fmt_eur(dep["scenario_net_with_deposit"], 0)},
]))
plot_deposit_simulation(dep)